# CEDA 토론 — 사전 준비된 전략 기반 테스트

`debate/example/` 폴더의 전략 문서(`pros`, `cons`)를 각 측 비공개 메모에 주입하여 토론을 실행한다.

**논제:** AI 채용 시스템이 인간의 판단보다 낫다

**특징:**
- 입론, CX 질문/답변, 반박이 모두 미리 세워둔 전략을 참고하여 생성됨
- 각 측은 상대 전략을 볼 수 없고 자기 전략만 참고 (State 격리)
- 전략 문서에는 상대측 반박 방향까지 준비되어 있어 재반박이 풍부해짐

## 1. Import 및 환경 확인

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, '../..')

from Agent_Structure.debate import (
    run_debate,
    stream_debate,
    create_debate,
    DebateConfig,
    DebateResult,
    CEDA_ROUNDS,
)
from Agent_Structure.config import settings

print(f"Provider: {settings.default_provider}")
print(f"Model: {settings.default_model}")
print(f"API Key 설정됨: {bool(settings.anthropic_api_key)}")
print(f"\nCEDA 라운드: {len(CEDA_ROUNDS)}개")

## 2. 전략 문서 로드

`example/pros`와 `example/cons` 파일을 읽어서 각 측 비공개 메모로 사용할 텍스트를 준비한다.

In [ ]:
EXAMPLE_DIR = Path('example')

aff_strategy = (EXAMPLE_DIR / 'pros').read_text(encoding='utf-8')
neg_strategy = (EXAMPLE_DIR / 'cons').read_text(encoding='utf-8')

print(f"긍정측 전략: {len(aff_strategy):,}자")
print(f"부정측 전략: {len(neg_strategy):,}자")

print("\n--- 긍정측 전략 미리보기 ---")
print(aff_strategy[:400] + '...')
print("\n--- 부정측 전략 미리보기 ---")
print(neg_strategy[:400] + '...')

## 3. 스트리밍 토론 실행 (전략 주입)

`aff_initial_notes`와 `neg_initial_notes` 파라미터로 전략 문서를 각 측 비공개 메모에 주입한다.

각 라운드에서 해당 측 에이전트는 **자기 전략만** 참고한다 (상대 전략은 차단됨).

In [ ]:
PROPOSITION = "AI 채용 시스템이 인간의 판단보다 낫다"

speaker_labels = {"affirmative": "긍정측", "negative": "부정측", "judge": "심판"}
type_labels = {
    "constructive": "입론",
    "cx_question": "교차조사 질문",
    "cx_answer": "교차조사 답변",
    "rebuttal": "반박",
    "verdict": "판정",
}

speeches = []

for speech in stream_debate(
    PROPOSITION,
    aff_initial_notes=aff_strategy,
    neg_initial_notes=neg_strategy,
):
    speeches.append(speech)
    speaker = speaker_labels.get(speech['speaker'], speech['speaker'])
    stype = type_labels.get(speech['speech_type'], speech['speech_type'])

    print(f"\n{'='*70}")
    print(f"[{speech['round_id']}] {speaker} — {stype}")
    print(f"{'='*70}")
    print(speech['content'])

print(f"\n\n총 {len(speeches)}개 발언 완료")

## 4. DebateConfig 사용 예시

여러 설정을 한꺼번에 관리할 때는 `DebateConfig`를 사용하면 편리하다.

In [ ]:
# DebateConfig로 한꺼번에 설정 주입 (실제 실행은 주석 처리)

# cfg = DebateConfig(
#     aff_initial_notes=aff_strategy,
#     neg_initial_notes=neg_strategy,
#     max_speech_chars=1000,  # 발언 길이 제한 완화
#     context_window=5,       # 컨텍스트 윈도우 확대
# )
#
# result = run_debate(PROPOSITION, config=cfg)
# print(result.format_transcript())
# print('\n=== 판정 ===')
# print(result.verdict)

print("DebateConfig 사용 예시 (주석 해제 후 실행)")

## 5. State 격리 검증

토론 과정에서 각 측의 비공개 메모가 상대측 transcript에 노출되지 않았는지 확인한다.

In [ ]:
# 긍정측 전략 문서에만 나오는 특이 문구 (부정측 문서엔 없음)
AFF_ONLY_MARKER = '긍정측 재반박 방향'
# 부정측 전략 문서에만 나오는 특이 문구 (긍정측 문서엔 없음)
NEG_ONLY_MARKER = '부정측 재반박 방향'

# 각 발언이 자기 측 전략만 반영했는지 점검 (직접 인용되지는 않았어야 함)
for speech in speeches:
    speaker = speech['speaker']
    content = speech['content']

    # 긍정측 발언에 부정측 전략 원문이 노출되었는지
    if speaker == 'affirmative' and NEG_ONLY_MARKER in content:
        print(f"⚠️ 격리 위반 의심: [{speech['round_id']}] 긍정측 발언에 부정측 전략 원문 노출")
        break
    # 부정측 발언에 긍정측 전략 원문이 노출되었는지
    if speaker == 'negative' and AFF_ONLY_MARKER in content:
        print(f"⚠️ 격리 위반 의심: [{speech['round_id']}] 부정측 발언에 긍정측 전략 원문 노출")
        break
else:
    print("✅ State 격리 OK — 상대 전략 원문이 발언에 노출되지 않았음")

print(f"\n긍정측 전략 포함 마커: '{AFF_ONLY_MARKER}'")
print(f"부정측 전략 포함 마커: '{NEG_ONLY_MARKER}'")

## 6. 심판 판정 별도 출력

In [ ]:
# transcript에서 판정만 추출
verdict_speeches = [s for s in speeches if s['speech_type'] == 'verdict']

if verdict_speeches:
    print('='*70)
    print('최종 판정')
    print('='*70)
    print(verdict_speeches[0]['content'])
else:
    print('아직 판정이 나오지 않음 (토론이 완료되지 않았을 수 있음)')

## 7. 전략 없이 실행한 결과와 비교 (옵션)

같은 논제를 전략 없이 실행해서 차이를 비교할 수 있다.

In [ ]:
# 전략 없이 실행 (비용 때문에 주석 처리)

# result_no_strategy = run_debate(PROPOSITION)
# print('=== 전략 없이 ===')
# print(result_no_strategy.format_transcript()[:2000])
# print('\n--- 판정 ---')
# print(result_no_strategy.verdict[:1000])

print("전략 유무 비교 (주석 해제 후 실행)")